## How to load PDF Files

In [ ]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader # will be written in later sections
)

In [ ]:
### PyPDFLoader
print("PyPDFLoader")

try:
    pypdf_loader = PyPDFLoader("data/pdf/google_infrastructure_whitepaper_fa.pdf")
    pypdf_loaded_docs = pypdf_loader.load()
    print(f"Loaded {len(pypdf_loaded_docs)} pages")
    print(f"The Spillted Page 1 Content loaded is {pypdf_loaded_docs[0].page_content[:100]}...")
    print(f"Metadata: {pypdf_loaded_docs[0].metadata}")
    # print(pypdf_loaded_docs)

except Exception as e:
    print(f"Oh No, we have encountered an error : {e}")

PyMuPDFLoader
Loaded 14 pages
The Spillted Page 1 Content loaded is Google Infrastructure SecurityDesign Overview
March 2022...
Metadata: {'producer': 'Skia/PDF m104 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Google Infrastructure Security Design Overview', 'source': 'data/pdf/google_infrastructure_whitepaper_fa.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}


In [ ]:
### PyPDFLoader (Fast and accurate)
print("PyMuPDFLoader")

try:
    pymupdf_loader = PyMuPDFLoader("data/pdf/google_infrastructure_whitepaper_fa.pdf")
    pymupdf_loaded_docs = pymupdf_loader.load()
    print(f"Loaded {len(pymupdf_loaded_docs)} pages")
    print(f"The Spillted Page 1 Content loaded is {pymupdf_loaded_docs[0].page_content[:100]}...")
    print(f"Metadata: {pymupdf_loaded_docs[0].metadata}")
    # print(pymupdf_loaded_docs)

except Exception as e:
    print(f"Oh No, we have encountered an error : {e}")

PyMuPDFLoader
Loaded 14 pages
The Spillted Page 1 Content loaded is Google Infrastructure Security
Design Overview
March 2022...
Metadata: {'producer': 'Skia/PDF m104 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'data/pdf/google_infrastructure_whitepaper_fa.pdf', 'file_path': 'data/pdf/google_infrastructure_whitepaper_fa.pdf', 'total_pages': 14, 'format': 'PDF 1.4', 'title': 'Google Infrastructure Security Design Overview', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}


### Difference between PyPDFLoader and PyMuPDFLoader

| Feature | `PyPDFLoader` | `PyMuPDFLoader` |
| :--- | :--- | :--- |
| **Engine / Under the Hood** | `pypdf` (Pure Python) | `PyMuPDF` / `fitz` (C-based MuPDF binding) |
| **Parsing Speed** | Slow to Moderate | **Extremely Fast** (10x–20x faster) |
| **Layout & Multi-Column Accuracy** | Basic; can scramble multi-column layouts and tables | High fidelity; preserves natural reading order & spacing |
| **Licensing** | **MIT License** (Free for commercial/closed-source use) | **AGPL-3.0 License** (Requires commercial license if proprietary) |
| **Dependencies** | Lightweight (Zero C-compilation dependencies) | Requires C extensions / binary dependencies |
| **Metadata & Extraction** | Basic page-level text extraction | Rich extraction (text, images, page coordinates, annotations) |
| **Best Used For...** | Enterprise apps with strict license compliance needs | High-volume ingestion & complex documents needing clean text |

> **Summary Rule of Thumb:** 
> * Use **`PyPDFLoader`** if you need safe, permissive licensing (MIT) and quick set-up without C-dependencies.
> * Use **`PyMuPDFLoader`** if you need fast ingestion and cleaner text extraction for better RAG retrieval quality.

### Handling PDF Challenges

* Can contain complex text, includes images, table, columns and may contain scanned images (requiring OCR) Often have extraction artificats


In [ ]:
Sample_PDF_information = """
Hey this is a sample PDF content that has lot of whitespaces




Yes, we will be writing pre-processing function for cleaning most of the PDF challenges on the below few lines of code


So yea
"""

def clean_PDF(text):

    # EXAMPLE using join and split - 
    # raw_text = "  RAG   pipelines   need \n clean   text.   "
    # # Step 1: text.split() creates a clean word list
    # words = raw_text.split()
    # # Result: ['RAG', 'pipelines', 'need', 'clean', 'text.']
    # # Step 2: " ".join() links them with a single space
    # clean_text = " ".join(words)
    # # Result: "RAG pipelines need clean text."

    # remove excessive whitespaces
    text = " ".join(text.split())

    # Fix ligtures
    
    return text

cleaned_PDF_Information = clean_PDF(Sample_PDF_information)
print("Before preprocessing - ")
print(repr(Sample_PDF_information))
print("\n After preprocessing - ")
print(repr(cleaned_PDF_Information))

Before preprocessing - 
'\nHey this is a sample PDF content that has lot of whitespaces\n\n\n\n\nYes, we will be writing pre-processing function for cleaning most of the PDF challenges on the below few lines of code\n\n\nSo yea\n'

 After preprocessing - 
'Hey this is a sample PDF content that has lot of whitespaces Yes, we will be writing pre-processing function for cleaning most of the PDF challenges on the below few lines of code So yea'


In [16]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [26]:
from langchain_core.documents import Document
from typing import List

class SmartPDFPreProcessor:
    def __init__(self,chunk_size=1000,chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separators = [" "]
        )

# def process_pdf(self, pdf_path: str) -> list[Document]:
# │   │           │     │         │    │  │       │
# │   │           │     │         │    │  │       └─ Contains LangChain Document objects
# │   │           │     │         │    │  └───────── ...a List...
# │   │           │     │         │    └──────────── Returns (ReturnType)
# │   │           │     │         └─────────────────── ...must be a string
# │   │           │     └───────────────────────────── Parameter 1 (pdf_path)...
# │   │           └─────────────────────────────────── Standard class instance reference
# │   └────────────────────────────────────────────── Method Name
# └────────────────────────────────────────────────── Method Keyword
    def process_pdf(self, pdf_path:str)->List[Document]:

        # Load PDF
        loader = PyPDFLoader(pdf_path)
        Pdf_pages = loader.load()

        # Process each page
        processed_chunks = []

        for page_num, page in enumerate(Pdf_pages):

            ## Clean Text
            cleaned_text = self._clean_text(page.page_content) # _clean_text is a private var which we will define in later stage

            ## Skip Nearly empty pages
            if len(cleaned_text.strip()) < 50: # Strip function removes method removes leading (start) and trailing (end) whitespace characters—including spaces, tabs (\t), and newlines (\n). 
                continue

            ## Create Chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(texts=[cleaned_text], 
                                                         metadatas=[{
                                                             **page.metadata,
                                                             "PDF_page" : page_num + 1,
                                                             "Total_PDF_pages" : len(Pdf_pages),
                                                             "char_count_of_current_page" : len(cleaned_text)
                                                         }])                
            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text:str)->str:

          # remove excessive whitespaces
            text = " ".join(text.split())    

            return text

    



In [27]:
PreProcessor = SmartPDFPreProcessor()

In [28]:
PreProcessor

In [29]:
# Process PDF
pdf_chunks = PreProcessor.process_pdf("data/pdf/google_infrastructure_whitepaper_fa.pdf")
print(f"Processed PDF file into number of {len(pdf_chunks)} smart chunks")

for key, value in pdf_chunks[0].metadata.items():
    print(f" {key} - {value}")

Processed PDF file into number of 36 smart chunks
 producer - Skia/PDF m104 Google Docs Renderer
 creator - PyPDF
 creationdate - 
 title - Google Infrastructure Security Design Overview
 source - data/pdf/google_infrastructure_whitepaper_fa.pdf
 total_pages - 14
 page - 0
 page_label - 1
 PDF_page - 1
 Total_PDF_pages - 14
 char_count_of_current_page - 56
